# Práctica: Reducción de dimensionalidad (Iris)

Objetivos:
- Clasificar Versicolor vs Virginica usando las 4 variables de Iris.
- Comparar tres enfoques:
  - Original: 4 variables.
  - Feature selection.
  - PCA: 73%, 96% y 99.5% de varianza explicada.
- Optimizar KNN, KDE y RDA con 10-fold CV usando AUC.
- Evaluar en test con matriz de confusión, métricas y ROC/AUC.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.neighbors import KNeighborsClassifier, KernelDensity
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)

## 1) Carga, filtrado y escalado

In [ ]:
iris = load_iris()

X_all = iris.data
y_all = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

mask = (y_all == 1) | (y_all == 2)
X = X_all[mask]
y_raw = y_all[mask]

# Reetiquetar: 0 = Versicolor, 1 = Virginica para evitar errores en las métricas próximas que funcionan mejor de forma binaria
y = np.where(y_raw == 1, 0, 1)

print("Clases: 0 =", target_names[1], "| 1 =", target_names[2])
print("X shape:", X.shape)
print("Variables:", feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=51, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train_s.shape, "Test:", X_test_s.shape)

## 2) Rejillas de hiperparámetros

In [ ]:
KNN_K = [1, 3, 5, 7, 9, 11, 13, 15]

KDE_KERNELS = ["gaussian", "tophat"]
KDE_BW = np.round(np.arange(0.1, 1.01, 0.1), 1).tolist()

RDA_ALPHAS = [0, 0.25, 0.5, 0.75, 1]

print("k-NN:", KNN_K)
print("KDE bandwidth:", KDE_BW)
print("RDA alpha:", RDA_ALPHAS)

## 3) Funciones auxiliares
Probabilidades para calcular AUC y curvas ROC.

In [ ]:
def proba_knn(k, Xtr, ytr, Xte):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(Xtr, ytr)
    return model.predict_proba(Xte)[:, 1]

def proba_rda(alpha, Xtr, ytr, Xte):
    model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage=alpha)
    model.fit(Xtr, ytr)
    return model.predict_proba(Xte)[:, 1]

def proba_kde(kernel, bw, Xtr, ytr, Xte):
    X0 = Xtr[ytr == 0]
    X1 = Xtr[ytr == 1]

    P0 = len(X0) / len(Xtr)
    P1 = len(X1) / len(Xtr)

    kde0 = KernelDensity(kernel=kernel, bandwidth=bw).fit(X0)
    kde1 = KernelDensity(kernel=kernel, bandwidth=bw).fit(X1)

    log0 = kde0.score_samples(Xte)
    log1 = kde1.score_samples(Xte)

    p0 = np.exp(log0) * P0
    p1 = np.exp(log1) * P1

    return p1 / (p0 + p1 + 1e-12)

def pred_from_proba(proba, threshold=0.5):
    return (proba >= threshold).astype(int)

def auc_10fold(get_proba_func, param, Xtr, ytr):
    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=51)
    proba_all = np.zeros(len(ytr))

    for tr_idx, va_idx in cv.split(Xtr, ytr):
        proba = get_proba_func(param, Xtr[tr_idx], ytr[tr_idx], Xtr[va_idx])
        proba_all[va_idx] = proba

    return roc_auc_score(ytr, proba_all)

## 4) Feature selection
Usamos 10-fold CV y AUC para escoger el subconjunto.

In [ ]:
def fs_proba_rda(alpha, Xtr, ytr, Xte):
    return proba_rda(alpha, Xtr, ytr, Xte)

def forward_selection_auc(Xtr, ytr, feature_names):
    selected = []
    remaining = list(range(Xtr.shape[1]))
    best_score = -1
    history = []

    while len(remaining) > 0:
        rows = []

        for feat in remaining:
            candidate = selected + [feat]
            X_candidate = Xtr[:, candidate]
            score = auc_10fold(fs_proba_rda, 0.25, X_candidate, ytr)

            rows.append({
                "candidate_feature": feature_names[feat],
                "features": [feature_names[i] for i in candidate],
                "auc": score,
                "feature_index": feat
            })

        df_step = pd.DataFrame(rows).sort_values("auc", ascending=False).reset_index(drop=True)
        best_row = df_step.iloc[0]

        if best_row["auc"] > best_score:
            selected.append(int(best_row["feature_index"]))
            remaining.remove(int(best_row["feature_index"]))
            best_score = best_row["auc"]

            history.append({
                "step": len(selected),
                "selected_feature": best_row["candidate_feature"],
                "features": [feature_names[i] for i in selected],
                "auc": best_score
            })
        else:
            break

    return selected, pd.DataFrame(history)

selected_features, df_fs_history = forward_selection_auc(X_train_s, y_train, feature_names)

selected_names = [feature_names[i] for i in selected_features]

print("Subconjunto seleccionado:")
print(selected_names)

display(df_fs_history)

## 5) PCA y scree plot

In [ ]:
pca_full = PCA()
pca_full.fit(X_train_s)

expl = pca_full.explained_variance_ratio_
cum_expl = np.cumsum(expl)

df_pca_info = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(expl))],
    "explained_var": expl,
    "cum_explained_var": cum_expl
})

display(df_pca_info)

plt.figure()
plt.bar(df_pca_info["PC"], df_pca_info["explained_var"], label="varianza")
plt.plot(df_pca_info["PC"], df_pca_info["cum_explained_var"], marker="o", label="acumulada")
plt.axhline(0.73, linestyle="--", color="gray")
plt.axhline(0.96, linestyle="--", color="gray")
plt.axhline(0.995, linestyle="--", color="gray")
plt.title("Scree plot PCA")
plt.ylabel("varianza explicada")
plt.legend()
plt.show()

In [ ]:
def n_components_for_threshold(cum_var, threshold):
    return int(np.searchsorted(cum_var, threshold) + 1)

n_pca_73 = n_components_for_threshold(cum_expl, 0.73)
n_pca_96 = n_components_for_threshold(cum_expl, 0.96)
n_pca_995 = n_components_for_threshold(cum_expl, 0.995)

print("Componentes PCA 73%:", n_pca_73)
print("Componentes PCA 96%:", n_pca_96)
print("Componentes PCA 99.5%:", n_pca_995)

## 6) Construcción de pipelines

In [ ]:
pipelines = {}

pipelines["Original_4vars"] = {
    "X_train": X_train_s,
    "X_test": X_test_s
}

pipelines["FeatureSelection"] = {
    "X_train": X_train_s[:, selected_features],
    "X_test": X_test_s[:, selected_features]
}

pca_73 = PCA(n_components=n_pca_73)
X_train_pca73 = pca_73.fit_transform(X_train_s)
X_test_pca73 = pca_73.transform(X_test_s)

pipelines["PCA_73"] = {
    "X_train": X_train_pca73,
    "X_test": X_test_pca73
}

pca_96 = PCA(n_components=n_pca_96)
X_train_pca96 = pca_96.fit_transform(X_train_s)
X_test_pca96 = pca_96.transform(X_test_s)

pipelines["PCA_96"] = {
    "X_train": X_train_pca96,
    "X_test": X_test_pca96
}

pca_995 = PCA(n_components=n_pca_995)
X_train_pca995 = pca_995.fit_transform(X_train_s)
X_test_pca995 = pca_995.transform(X_test_s)

pipelines["PCA_995"] = {
    "X_train": X_train_pca995,
    "X_test": X_test_pca995
}

for name, data in pipelines.items():
    print(name, "->", data["X_train"].shape[1], "variables/componentes")

## 7) Optimización con 10-fold CV y AUC
Para cada pipeline y algoritmo elegimos el hiperparámetro con mayor AUC medio.

In [ ]:
def optimize_knn(Xtr, ytr):
    rows = []
    for k in KNN_K:
        score = auc_10fold(lambda p, Xa, ya, Xb: proba_knn(p, Xa, ya, Xb), k, Xtr, ytr)
        rows.append({"algorithm": "kNN", "param": k, "auc_cv": score})
    df = pd.DataFrame(rows)
    return df.sort_values("auc_cv", ascending=False).iloc[0], df

def optimize_rda(Xtr, ytr):
    rows = []
    for a in RDA_ALPHAS:
        score = auc_10fold(lambda p, Xa, ya, Xb: proba_rda(p, Xa, ya, Xb), a, Xtr, ytr)
        rows.append({"algorithm": "RDA", "param": a, "auc_cv": score})
    df = pd.DataFrame(rows)
    return df.sort_values("auc_cv", ascending=False).iloc[0], df

def optimize_kde(Xtr, ytr):
    rows = []
    for ker in KDE_KERNELS:
        for bw in KDE_BW:
            param = (ker, bw)
            score = auc_10fold(lambda p, Xa, ya, Xb: proba_kde(p[0], p[1], Xa, ya, Xb), param, Xtr, ytr)
            rows.append({"algorithm": "KDE", "param": param, "auc_cv": score})
    df = pd.DataFrame(rows)
    return df.sort_values("auc_cv", ascending=False).iloc[0], df

In [ ]:
best_rows = []
all_cv_results = {}

for pname, pdata in pipelines.items():
    Xtr = pdata["X_train"]

    best_knn, df_knn_cv = optimize_knn(Xtr, y_train)
    best_rda, df_rda_cv = optimize_rda(Xtr, y_train)
    best_kde, df_kde_cv = optimize_kde(Xtr, y_train)

    all_cv_results[(pname, "kNN")] = df_knn_cv
    all_cv_results[(pname, "RDA")] = df_rda_cv
    all_cv_results[(pname, "KDE")] = df_kde_cv

    for best in [best_knn, best_rda, best_kde]:
        best_rows.append({
            "pipeline": pname,
            "algorithm": best["algorithm"],
            "best_param": best["param"],
            "auc_cv": best["auc_cv"]
        })

df_best = pd.DataFrame(best_rows)
display(df_best.sort_values(["pipeline", "algorithm"]))

## 8) Entrenamiento final y evaluación en test

In [ ]:
def final_proba(algorithm, param, Xtr, ytr, Xte):
    if algorithm == "kNN":
        return proba_knn(int(param), Xtr, ytr, Xte)
    if algorithm == "RDA":
        return proba_rda(float(param), Xtr, ytr, Xte)
    if algorithm == "KDE":
        ker, bw = param
        return proba_kde(ker, bw, Xtr, ytr, Xte)
    raise ValueError("Algoritmo no reconocido")

test_rows = []
roc_data = []

for _, row in df_best.iterrows():
    pname = row["pipeline"]
    alg = row["algorithm"]
    param = row["best_param"]

    Xtr = pipelines[pname]["X_train"]
    Xte = pipelines[pname]["X_test"]

    proba = final_proba(alg, param, Xtr, y_train, Xte)
    y_pred = pred_from_proba(proba)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    sens = rec
    spec = tn / (tn + fp + 1e-12)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, proba)

    test_rows.append({
        "pipeline": pname,
        "algorithm": alg,
        "best_param": param,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "sensibilidad": sens,
        "especificidad": spec,
        "f1": f1,
        "auc": auc
    })

    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_data.append({
        "pipeline": pname,
        "algorithm": alg,
        "best_param": param,
        "fpr": fpr,
        "tpr": tpr,
        "auc": auc
    })

df_test = pd.DataFrame(test_rows)
display(df_test.sort_values("auc", ascending=False))

## 9) Curvas ROC en test
(Una figura por algoritmo para comparar los pipelines sin sobrecargar)

In [ ]:
for alg in ["kNN", "KDE", "RDA"]:
    plt.figure()

    for r in roc_data:
        if r["algorithm"] == alg:
            label = f'{r["pipeline"]} AUC={r["auc"]:.3f}'
            plt.plot(r["fpr"], r["tpr"], label=label)

    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.title(f"ROC en test - {alg}")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.legend()
    plt.show()

## 10) Resumen

##### Subconjunto final:
Feature selection selecciona tres variables:
- petal width (cm)
- petal length (cm)
- sepal length (cm)

Por tanto, la variable descartada es sepal width (cm).


##### PCA:
- PCA 73% usa 1 componente.
- PCA 96% usa 3 componentes.
- PCA 99.5% usa 4 componentes.

El scree plot muestra que PC1 explica el 74.62% de la varianza. Por eso PCA 73% usa 1 componente.
Para llegar al 96% hacen falta 3 componentes, ya que con PC1+PC2+PC3 se alcanza el 98.21%.
Para llegar al 99.5% hacen falta las 4 componentes, porque con 3 componentes se queda en 98.21% y solo con PC4 se alcanza el 100%.

##### Hiperparámetros elegidos:
Los mejores parámetros mantienen valores altos de AUC en casi todos los pipelines. En Original_4vars se obtiene KNN con k = 11, KDE gaussian con bandwidth = 0.7 y RDA con alpha = 0.25. En FeatureSelection se obtiene kNN con k = 15, KDE gaussian con bandwidth = 0.3 y RDA con alpha = 0.25. En PCA 73% los resultados empeoran y se usan k = 15, KDE gaussian con bandwidth = 1.0 y RDA con alpha = 0.0. En PCA 96% se usan k=11, KDE gaussian con bandwidth = 0.8 y RDA con alpha = 0.0. En PCA 99.5% se recupera prácticamente el caso de 4 variables, con k = 11, KDE gaussian con bandwidth = 0.7 y RDA con alpha = 0.25.

##### Resultados en test:
El mejor resultado global lo obtiene FeatureSelection con KNN, ya que alcanza accuracy = 1.00, precision = 1.00, recall/sensibilidad = 1.00, especificidad = 1.00, F1 = 1.00 y AUC = 1.00. También hay modelos con AUC = 1.00, como RDA con Original_4vars y RDA con PCA 99.5%, pero su accuracy queda en 0.9667.

##### Comparación entre pipelines:
Usar las 4 variables funciona muy bien, especialmente con RDA. FeatureSelection mejora la simplicidad porque elimina una variable y mantiene incluso el mejor rendimiento en test con KNN. PCA 73% pierde demasiada información al usar solo una componente y por eso baja claramente el rendimiento. PCA 96% funciona bastante bien con 3 componentes y PCA 99.5% casi reproduce el rendimiento de usar las 4 variables.

##### Conclusión:
Si se prioriza el mejor rendimiento en test, recomendaría FeatureSelection + kNN, porque obtiene todas las métricas perfectas en este split. Si se prioriza simplicidad e interpretación, también recomendaría FeatureSelection, ya que usa solo 3 variables originales y evita transformar los datos como hace la estructura de PCA. PCA solo es recomendable si interesa comprimir variables, pero en este caso PCA 73% reduce demasiado y empeora la generalización, por lo que sigo recomendando lo planteado al inicio del parrafo.